In [12]:
import numpy as np
from scipy.optimize import *

In [13]:
# Sample hourly outdoor temperatures (24 hours)
outdoor_temps = [10, 9, 8, 7, 7, 6, 7, 9, 12, 15, 18, 20, 22, 23, 24, 24, 23, 22, 20, 18, 16, 14, 12, 11]

# Desired indoor temperature
desired_temp = 22  # in degrees Celsius
temp_range = 2  # permissible range from the desired temperature
temp_inertia = 1 # thermal inertia parameter (max hourly temperature change)
rate = 0.5  # hypothetical energy rate

In [16]:
# Energy consumption function
def energy_consumption(indoor_temps, outdoor_temps, rate):
    return sum(rate * abs(indoor - outdoor) 
               for indoor, outdoor in zip(indoor_temps, outdoor_temps))

def objective(indoor_temps):
    return energy_consumption(indoor_temps, outdoor_temps, rate)

# Constraints

# Comfort range constraints
def constraint_upper(indoor_temps):
    return (desired_temp + temp_range) - indoor_temps

def constraint_lower(indoor_temps):
    return indoor_temps - (desired_temp - temp_range)

# Thermal inertia constraints
# |T(i+1) - T(i)| <= temp_inertia

# Implemented as:
#   T(i+1) - T(i) <= temp_inertia
#   -(T(i+1) - T(i)) <= temp_inertia
def constraint_rate_up(indoor_temps):
    return np.array([temp_inertia - (indoor_temps[i+1] - indoor_temps[i]) 
                     for i in range(23)])

def constraint_rate_down(indoor_temps):
    return np.array([temp_inertia + (indoor_temps[i+1] - indoor_temps[i]) 
                     for i in range(23)])


# Optimization setup

# Initial guess (keeping indoor temperature constant at desired_temp)
initial_guess = [desired_temp] * 24

# Define the bounds for each hour's temperature
bounds = [(desired_temp - temp_range, desired_temp + temp_range)] * 24

# Optimization
cons = [
    {'type': 'ineq', 'fun': constraint_upper},
    {'type': 'ineq', 'fun': constraint_lower},
    {'type': 'ineq', 'fun': constraint_rate_up},
    {'type': 'ineq', 'fun': constraint_rate_down}
]

result = minimize(objective,
                  initial_guess,
                  method='SLSQP',
                  bounds=bounds,
                  constraints=cons)
# Results
if result.success:
    optimized_temps = result.x
    print("Optimized indoor temperatures:", optimized_temps)
    print("Total energy consumption:", objective(optimized_temps))
else:
    print("Optimization failed:", result.message)

Optimized indoor temperatures: [20.         20.         20.         20.         20.         20.
 20.         20.         20.         20.         20.         20.99999998
 21.99999998 22.99999998 23.99999998 23.99999998 22.99999998 21.99999998
 20.99999998 20.         20.         20.         20.         20.        ]
Total energy consumption: 71.50000004856955
